# 03 ACS — Drizzle FLC Files

**Inputs:** FLC files in `data_acs/raw/<target>/` — CTE-corrected by CalACS (full-frame)
or by `02_cte_correct_acs.ipynb` (subarray targets). All 5 targets are processed.

**Outputs:** `data_acs/processed/<target>/drizzled/<target>_<filter>_drz.fits`

CR rejection is fully enabled (driz_separate → median → blot → driz_cr) — required for ACS CCD data.  
All drizzle parameters read from `config/wfc3_ir_drizzle_params.yaml`.

In [ ]:
%matplotlib inline
import os
import shutil
import yaml
from pathlib import Path
from collections import defaultdict
from astropy.io import fits
from astropy import wcs
from astropy.visualization import ImageNormalize, LogStretch
import matplotlib.pyplot as plt
from drizzlepac import astrodrizzle
from IPython.display import clear_output, display

import subprocess
REPO_ROOT = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel']).decode().strip())
RAW_DIR  = REPO_ROOT / 'data_acs' / 'raw'
PROC_DIR = REPO_ROOT / 'data_acs' / 'processed'
CONFIG   = REPO_ROOT / 'config' / 'wfc3_ir_drizzle_params.yaml'

drizzle_cfg = yaml.safe_load(open(CONFIG))['acs_drizzle']
print('Config loaded.')
print(f'RAW_DIR  : {RAW_DIR}')
print(f'PROC_DIR : {PROC_DIR}')

In [ ]:
# Remove any AstroDrizzle intermediate files left in raw/ directories from
# previous failed runs. These are safe to delete — they are recreated on each
# drizzle run and should live in the output directory, not alongside the inputs.

INTERMEDIATE_PATTERNS = [
    '*_single_sci.fits', '*_single_wht.fits',
    '*_blt.fits', '*_crmask.fits',
    '*_final_mask.fits', '*_single_mask.fits',
]

removed = []
for pattern in INTERMEDIATE_PATTERNS:
    for f in RAW_DIR.rglob(pattern):
        f.unlink()
        removed.append(f.relative_to(RAW_DIR))

if removed:
    print(f'Removed {len(removed)} stray intermediate files from data_acs/raw/:')
    for p in sorted(removed):
        print(f'  {p}')
else:
    print('No stray intermediate files found.')

In [ ]:
# Build input groups: one list of FLC files per (target, filter) pair

def get_filter(hdr):
    f1, f2 = hdr.get('FILTER1', ''), hdr.get('FILTER2', '')
    return f1 if 'CLEAR' not in f1 else f2

groups = defaultdict(list)
for f in sorted(RAW_DIR.glob('*/*_flc.fits')):
    if f.name.startswith('hst_'):
        continue
    target = f.parent.name
    with fits.open(f) as hdul:
        filt = get_filter(hdul[0].header)
    groups[(target, filt)].append(str(f.resolve()))

print(f'Found {len(groups)} target/filter groups to drizzle:\n')
for (target, filt), files in sorted(groups.items()):
    print(f'  {target}  {filt}  {len(files)} FLC files')

In [ ]:
original_dir = os.getcwd()

for (target, filt), files in sorted(groups.items()):
    out_dir = (PROC_DIR / target / 'drizzled').resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    output_name = f'{target}_{filt}'
    # AstroDrizzle produces _drc.fits (not _drz.fits) when inputs are FLC files
    drc_out = out_dir / f'{output_name}_drc.fits'

    if drc_out.exists():
        print(f'{target} {filt}: already drizzled, skipping')
        continue

    print(f'Drizzling {target} {filt} ({len(files)} FLC files) ...')
    os.chdir(str(out_dir))
    try:
        astrodrizzle.AstroDrizzle(
            files,
            output=output_name,
            context=False,
            build=True,
            preserve=False,
            skysub=True,
            skymethod=drizzle_cfg['skymethod'],
            driz_separate=True,
            median=True,
            blot=True,
            driz_cr=True,
            final_wcs=drizzle_cfg['final_wcs'],
            final_rot=0.,
            final_scale=drizzle_cfg['final_scale'],
            final_pixfrac=drizzle_cfg['final_pixfrac'],
            final_kernel=drizzle_cfg['final_kernel'],
            driz_sep_bits=drizzle_cfg['driz_sep_bits'],
            final_bits=drizzle_cfg['final_bits'],
            combine_type=drizzle_cfg['combine_type'],
        )
    finally:
        os.chdir(original_dir)

    clear_output(wait=True)
    status = 'OK' if drc_out.exists() else 'FAILED — no DRC produced'
    print(f'  {target} {filt} → {status}')

print('\nAll targets drizzled.')

In [ ]:
# Inspect drizzled science and weight images for all targets and filters
vmin, vmax = -0.05, 100
GLIKMAN_DIR = Path('/home/daltshuler/RedQuasarResearch/GlikmanFIles')

for quasar_dir in sorted(PROC_DIR.iterdir()):
    drz_dir = quasar_dir / 'drizzled'
    if not drz_dir.exists():
        continue
    # AstroDrizzle names output _drc.fits when inputs are FLC (CTE-corrected) files
    drc_files = sorted(drz_dir.glob('*_drc.fits'))
    if not drc_files:
        continue

    target = quasar_dir.name

    for drc_file in drc_files:
        filt = drc_file.stem.replace(f'{target}_', '').replace('_drc', '')

        with fits.open(drc_file) as hdu:
            im_wcs = wcs.WCS(hdu[1].header)
            sci    = hdu[1].data
            wht    = hdu[2].data

        norm = ImageNormalize(sci, vmin=vmin, vmax=vmax, stretch=LogStretch())

        fig, axes = plt.subplots(1, 2, figsize=(20, 15), subplot_kw={'projection': im_wcs})
        fig.suptitle(f'{target} — {filt}', fontsize=20)

        axes[0].imshow(sci, norm=norm, cmap='gray', origin='lower')
        axes[0].set_title('Science (SCI)', fontsize=16)

        axes[1].imshow(wht, cmap='gray', origin='lower')
        axes[1].set_title('Weight (WHT)', fontsize=16)

        for ax in axes:
            ax.tick_params(labelsize=13)

        plt.tight_layout()

        out_png = drz_dir / f'{target}_{filt}_drc_inspect.png'
        fig.savefig(out_png, dpi=150, bbox_inches='tight')

        glik_out = GLIKMAN_DIR / target
        glik_out.mkdir(parents=True, exist_ok=True)
        fig.savefig(glik_out / out_png.name, dpi=150, bbox_inches='tight')

        display(fig)
        plt.close(fig)